# Finding Best Drop Combination

Vamos a encontrar la combinación de columnas que minimiza el **SMD** en el *propensity score matching*.

## Contexto

Hola Mariano. Estoy seguro de que en algún momento se te va a olvidar qué estás haciendo aquí, así que voy a dejar documentado qué estoy haciendo y por qué.

Hoy estás retomando tu tesis después de aproximadamente **3 meses** sin avanzar. En este punto ya no recuerdas exactamente dónde te habías quedado.

Te quedaste en la **redacción de las pruebas de balance**.

Las pruebas de balance sirven para demostrar que la **selección de unidades de control fue adecuada**.  
"Adecuada" significa que las **diferencias en las distribuciones de las covariables entre tratamiento y control se redujeron lo más posible**.  

Esto permite argumentar que las unidades de tratamiento y de control son **estadísticamente comparables**.

### Qué se tiene que demostrar

Hay dos cosas principales que tienes que mostrar:

#### 1. Balance de covariables

Debes demostrar que, después del matching, las **diferencias entre covariables se reducen significativamente** entre los grupos de tratamiento y control.

Esto normalmente se muestra con **love plots**, utilizando el **SMD (Standardized Mean Difference)**.

#### 2. Existencia de *Common Support*

También tienes que demostrar que existen **suficientes unidades de control comparables para las unidades tratadas**.

Esto se conoce como **common support**.

Para mostrarlo, normalmente se grafican las **distribuciones de los propensity scores** para:

- Unidades tratadas
- Unidades de control

Si existen unidades tratadas para las cuales **no hay unidades de control con un propensity score similar**, entonces esas unidades deben **excluirse del universo de análisis**.

En tu caso **esto no sucede**, porque tienes muchas unidades de control, pero **de todas formas tienes que mostrarlo**.

### Problema actual

Todas estas pruebas las habías hecho **únicamente para las unidades definidas con el grid cuadrado**, y **no para las unidades circulares**.

Por lo tanto, ahora necesitas:

- Generar **love plots** para las unidades circulares.
- Mostrar **common support** también para esas unidades.

### Qué hace este script

En este script estás **buscando qué variables eliminar del cálculo del propensity score** para obtener el mejor matching posible.

Más específicamente:

Estás intentando encontrar **la combinación de covariables que debe eliminarse** para que **la suma total de los SMD sea lo más baja posible**.

Ese es el objetivo de este script.

Por favor no lo olvides.

## Código

In [1]:
import os
import sys
import pickle
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Agregar el directorio base (scripts) al path para importar las clases
sys.path.append(os.path.abspath('..'))

from ps_features_builder import PSFeaturesBuilder
from ps_matching import PSMatching

In [3]:
# Definición de rutas a los archivos
PATH_DATA = '../../data/'
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet')
}

In [4]:
radios = [
    (37.5, 25),
    (50, 15),
    (100, 20),
    (150, 15),
    (200, 20),
    (250, 25),
]

In [25]:
radio, n_circles = radios[2]

ps_builder = PSFeaturesBuilder(
    paths=PATHS,
    grid_type='circular',
    # grid_size=radio, # grid_size parameter is currently for rectangular, but circular radius works here too if we want to log it or we just use circle_radius
    circle_radius=radio,
    n_circles=n_circles
)
ps_builder.build()

# 2. Matching
psm = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type='circular',
    grid_size=radio # Importante para excluir features en el modelo Logit según el dict DROP_COLUMNS_DICT_CIRCULAR
)
# psm.build(matching_method='nearest', n_matches=1, replace=False)

In [28]:
psm.find_best_drop_columns(
    max_subset_size=3,
)

Evaluando 987 combinaciones postuladas...


Iterando combinaciones drop_columns:   0%|          | 0/987 [00:00<?, ?it/s]

{'best_drop_columns': ['via_primaria', 'road_length_m', 'distance_to_station'],
 'min_smd_sum': np.float64(0.7719752937023379),
 'resultados':                                           drop_columns   sum_smd
 559  [via_primaria, road_length_m, distance_to_stat...  0.771975
 789      [road_length_m, distance_to_station, std_min]  1.052911
 566            [via_primaria, road_length_m, mean_fcs]  1.155668
 768  [road_length_m, mean_afluencia_mensual, distan...  1.157248
 468  [both_directions, road_length_m, distance_to_s...  1.197878
 ..                                                 ...       ...
 700                  [via_acc_cont, mean_fcs, std_fcs]  8.715787
 696                 [via_acc_cont, mean_pic, mean_fcs]  8.800625
 650   [via_acc_cont, mean_afluencia_mensual, mean_min]  8.850082
 4                                       [via_acc_cont]  9.121353
 676               [via_acc_cont, mean_total, mean_pic]  9.537721
 
 [987 rows x 2 columns]}

In [29]:
ps_builder.circle_radius

100

In [26]:
# guardamos el builder
with open(os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder", f"circular_{radio}_{n_circles}.pkl"), "wb") as f:
    pickle.dump(ps_builder, f)